# Gemma: huấn luyện và đánh giá trích xuất

Gemma `google/gemma-4-E4B-it` trích xuất tin tuyển dụng; Ridge/Random Forest dự đoán lương ở notebook 02. **Run All với mặc định chỉ kiểm tra và in kế hoạch**, không train hoặc gọi model suy luận.

Đã train thật qua CLI: full 364 bước và các lượt cải tiến theo xác nhận người dùng. Xem `docs/GEMMA_TUNING.md`, `outputs/metrics/gemma_tuning/REPORT.md` và kết quả cuối `FINAL_REPORT.md` sau khi chốt adapter. Không chọn model theo test hoặc chỉ theo loss.

Nhãn silver: 364/72/82 mẫu từ 122/25/28 tin nguồn; các biến thể không độc lập. Chưa chứng minh khả năng đọc mọi tin tự do hoặc tư vấn nghề nghiệp.

In [ ]:
from pathlib import Path
import json
import subprocess
import sys
ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
assert (ROOT / "configs/gemma_train.yaml").exists()
PYTHON = sys.executable

def run_script(*args):
    subprocess.run([PYTHON, *args], cwd=ROOT, check=True)

print("Workspace:", ROOT)

## 1. Kiểm tra dữ liệu và trọng số
Chỉ xem ví dụ train. Không dùng test để điều chỉnh prompt, tham số hoặc chọn checkpoint. Trọng số lớn không lưu lên Git; xem manifest và hướng dẫn tải trong README.

In [ ]:
run_script("scripts/validate_sft.py", "--strict-extraction")
manifest = json.loads((ROOT / "data/finetune/manifest.json").read_text())
weights = json.loads((ROOT / "outputs/gemma_preparation/weights.json").read_text())
print(json.dumps(manifest["splits"], ensure_ascii=False, indent=2))
print(json.dumps(weights, ensure_ascii=False, indent=2))
example = json.loads((ROOT / "data/finetune/train.jsonl").read_text().splitlines()[0])
print(json.dumps(example, ensure_ascii=False, indent=2))

## 2. Xem kế hoạch full, chưa train
LoRA MLX 4-bit, 4 lớp cuối, rank 8, batch 1, 364 updates = 1 epoch đủ 364 mẫu train. Learning rate 1e-4, context 1024, mask prompt, gradient checkpoint. Lệnh full bắt đầu từ base; các config `gemma_tune_*.yaml` ghi adapter cha cho các lượt cải tiến riêng.

Lưu checkpoint mỗi 100 bước. Loss không thay thế accuracy/F1. Không chạy GPU training và evaluation đồng thời.

In [ ]:
run_script("scripts/gemma_full_train.py", "--epochs", "1")

## 3. Chỉ khởi chạy sau khi bạn xác nhận
Chỉ điền chuỗi bên dưới sau khi đã đồng ý kế hoạch. Mặc định ô này bỏ qua huấn luyện. Log lưu tại `outputs/training_logs/gemma_<UTC>/`; adapter lưu thư mục riêng theo lượt để bảo toàn checkpoint.

In [ ]:
GEMMA_TRAINING_CONFIRMATION = ""  # Chỉ đổi khi chính bạn muốn bắt đầu một lượt mới.
if GEMMA_TRAINING_CONFIRMATION == "TOI_XAC_NHAN_TRAIN_GEMMA":
    run_script("scripts/gemma_full_train.py", "--epochs", "1", "--confirm-training")
else:
    print("KHÔNG train. Muốn suy luận, dùng make gemma-infer-adapter.")

## 4. Đánh giá trên validation: suy luận, không train
Bật inference khi sẵn sàng, dùng thư mục output mới. Mặc định đọc adapter được chọn từ `selection.json`; không lấy run mới nhất hoặc loss thấp nhất. Mỗi evaluation dùng temp 0, seed 42, max 256 token và không thinking.

Metric thô: JSON/schema validity, accuracy 8 trường, Skill micro F1. Không sửa JSON, alias hoặc duplicate trước khi chấm. Trường thiếu tính sai. Numeric tolerance 1e-6 theo đơn vị gốc; skills exact theo tập chuỗi gốc, F1 theo tập tên casefold. Nhãn silver và các biến thể phụ thuộc giới hạn ý nghĩa kết quả.

In [ ]:
RUN_INFERENCE = False
selection_path = ROOT / "outputs/metrics/gemma_tuning/selection.json"
selection = json.loads(selection_path.read_text()) if selection_path.exists() else {}
ADAPTER = selection.get("selected_adapter", "adapters/gemma-jobs/20261017T101925Z")
EVAL_RUN = "manual_validation_001"  # Đổi tên nếu thư mục đã tồn tại.
if RUN_INFERENCE:
    run_script("scripts/evaluate_gemma.py", "--generate", "--split", "valid",
               "--adapter", ADAPTER,
               "--output", f"outputs/metrics/gemma_{EVAL_RUN}_adapter_valid")
else:
    print("Chưa chạy suy luận. Adapter theo validation:", ADAPTER)

## 5. Đọc kết quả cuối và ghi báo cáo
Sau khi chọn bằng validation, chỉ đánh giá test cho adapter cố định; không train tiếp dựa trên lỗi test. Kết quả đã có được đọc lại bên dưới, không chạy model thêm. Dẫn `metrics.json`, `predictions.jsonl`, checksum, revision và log vào báo cáo.

Không ghép lương năm/tháng/ngoại tệ vào một đơn vị giả định. Kết quả Gemma là trích xuất dữ liệu, không phải sai số dự đoán lương của notebook 02.

In [ ]:
report = ROOT / "outputs/metrics/gemma_tuning/FINAL_REPORT.md"
if report.exists():
    from IPython.display import Markdown, display
    display(Markdown(report.read_text()))
else:
    print("Chưa chốt đánh giá cuối; xem REPORT.md và selection.json để theo dõi validation.")